In [1]:
# ========= 标准库 =========
import os
import time
import logging
from collections import Counter, defaultdict
from itertools import cycle
from typing import Dict, List

# ========= 科学计算 / 数据处理 =========
import joblib
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.metrics import (
    auc,
    classification_report,
    confusion_matrix,
    roc_curve,
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample

# ========= 深度学习 & 机器学习 =========
import tensorflow as tf
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.datasets import mnist
from tensorflow.keras.initializers import HeNormal, glorot_uniform
from tensorflow.keras.layers import (
    Add,
    AveragePooling1D,
    BatchNormalization,
    Conv1D,
    Dense,
    Flatten,
    Input,
    MaxPooling1D,
    ZeroPadding1D,
    Activation,
)
from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from keras.metrics import Precision, Recall  # 可替换为 tf.keras.metrics.*

# ========= 可视化 =========
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
label_col = "Attack_type"

In [3]:
df = pd.read_csv('./preprocessed_DNN.csv', low_memory=False)
#df = pd.read_csv('./downsampled_preprocessed_DNN.csv', low_memory=False)
feat_cols = list(df.columns)

feat_cols.remove(label_col)
feat_cols

skip_list = ["icmp.unused", "http.tls_port", "dns.qry.type", "mqtt.msg_decoded_as", "Attack_label"]
df.drop(skip_list, axis=1, inplace=True)

feat_cols = list(df.columns)
feat_cols.remove(label_col)

In [4]:
df

,arp.opcode,arp.hw.size,icmp.checksum,icmp.seq_le,http.content_length,http.response,tcp.ack,tcp.ack_raw,tcp.checksum,tcp.connection.fin,...,mqtt.conack.flags-1471198,mqtt.conack.flags-1471199,mqtt.conack.flags-1574358,mqtt.conack.flags-1574359,mqtt.protoname-0,mqtt.protoname-0.0,mqtt.protoname-MQTT,mqtt.topic-0,mqtt.topic-0.0,mqtt.topic-Temperature_and_Humidity
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.647490e+09,1594.0,0.0,...,False,False,False,False,True,False,False,True,False,False
1,0.0,0.0,0.0,0.0,0.0,0.0,59.0,3.411509e+09,54649.0,1.0,...,False,False,False,False,True,False,False,True,False,False
2,0.0,0.0,0.0,0.0,0.0,0.0,5.0,1.099419e+09,31572.0,0.0,...,False,False,False,False,True,False,False,False,False,True
3,0.0,0.0,0.0,0.0,0.0,0.0,59.0,1.185254e+09,54569.0,0.0,...,False,False,False,False,True,False,False,True,False,False
4,0.0,0.0,0.0,0.0,0.0,0.0,56.0,1.795444e+09,36563.0,0.0,...,False,False,False,False,True,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1909666,0.0,0.0,0.0,0.0,0.0,0.0,115678289.0,1.254530e+09,30876.0,0.0,...,False,False,False,False,True,False,False,True,False,False
1909667,0.0,0.0,0.0,0.0,0.0,0.0,56.0,1.594269e+09,11256.0,0.0,...,False,False,False,False,True,False,False,True,False,False
1909668,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.213215e+09,59837.0,0.0,...,False,False,False,False,False,True,False,False,True,False
1909669,0.0,0.0,0.0,0.0,0.0,0.0,479.0,4.273690e+09,7664.0,0.0,...,False,False,False,False,False,True,False,False,True,False


In [5]:
df.Attack_type.value_counts()

Attack_type
Normal                   1363998
DDoS_UDP                  121567
DDoS_ICMP                  67939
SQL_injection              50826
DDoS_TCP                   50062
Vulnerability_scanner      50026
Password                   49933
DDoS_HTTP                  48544
Uploading                  36807
Backdoor                   24026
Port_Scanning              19977
XSS                        15066
Ransomware                  9689
Fingerprinting               853
MITM                         358
Name: count, dtype: int64

# Preprocessing (normalization and padding values)

In [6]:
# Z-score normalization
features = df.dtypes[df.dtypes != 'object'].index
df[features] = df[features].apply(
    lambda x: (x - x.mean()) / (x.std()))
# Fill empty values by 0
df = df.fillna(0)

In [7]:
label_encoder = LabelEncoder()
df['Attack_type'] = label_encoder.fit_transform(df['Attack_type'])


In [8]:
df.Attack_type.value_counts()

Attack_type
7     1363998
4      121567
2       67939
11      50826
3       50062
13      50026
8       49933
1       48544
12      36807
0       24026
9       19977
14      15066
10       9689
5         853
6         358
Name: count, dtype: int64

In [9]:
X_fs = df.drop([label_col], axis=1)
y = df[label_col]

In [10]:
X_fs

,arp.opcode,arp.hw.size,icmp.checksum,icmp.seq_le,http.content_length,http.response,tcp.ack,tcp.ack_raw,tcp.checksum,tcp.connection.fin,...,mqtt.conack.flags-1471198,mqtt.conack.flags-1471199,mqtt.conack.flags-1574358,mqtt.conack.flags-1574359,mqtt.protoname-0,mqtt.protoname-0.0,mqtt.protoname-MQTT,mqtt.topic-0,mqtt.topic-0.0,mqtt.topic-Temperature_and_Humidity
0,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149153,-0.061580,-1.361015,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,0.700561,-0.632498,-0.213182
1,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149152,1.297649,1.229695,2.984767,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,0.700561,-0.632498,-0.213182
2,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149153,-0.483885,0.102830,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,-1.427427,-0.632498,4.690833
3,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149152,-0.417746,1.225788,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,0.700561,-0.632498,-0.213182
4,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149152,0.052423,0.346544,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,0.700561,-0.632498,-0.213182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1909666,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,0.502600,-0.364367,0.068844,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,0.700561,-0.632498,-0.213182
1909667,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149152,-0.102589,-0.889213,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,0.700563,-0.632498,-0.213186,0.700561,-0.632498,-0.213182
1909668,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149153,0.374328,1.483028,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,-1.427422,1.581031,-0.213186,-1.427427,1.581031,-0.213182
1909669,-0.003437,-0.003618,-0.166331,-0.179964,-0.053767,-0.131792,-0.149150,1.961984,-1.064613,-0.335034,...,-0.001023,-0.001023,-0.002171,-0.002171,-1.427422,1.581031,-0.213186,-1.427427,1.581031,-0.213182


# Solve class-imbalance by SMOTE

In [11]:
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
import pandas as pd

def hybrid_resample(X, y, upsample_targets=[5, 6], total_per_class=9689, random_state=42):
    # 创建 DataFrame
    df = pd.DataFrame(X)
    df['label'] = y  # 假设标签列为 'label'，可以根据实际情况修改
    
    # 1. 分离需要上采样的类和其他类
    df_upsample = df[df['label'].isin(upsample_targets)]
    df_other = df[~df['label'].isin(upsample_targets)]
    
    # 2. 对需要上采样的类分别上采样到 total_per_class
    df_list = []
    for cls in upsample_targets:
        df_cls = df_upsample[df_upsample['label'] == cls]
        df_cls_up = resample(
            df_cls,
            replace=True,  # 允许重复采样
            n_samples=total_per_class,
            random_state=random_state
        )
        df_list.append(df_cls_up)
    
    # 合并上采样的类别
    df_upsampled = pd.concat(df_list, axis=0)
    
    # 4. 合并上采样后的类别和未变化的其他类
    df_final = pd.concat([df_upsampled, df_other], axis=0).sample(frac=1.0, random_state=random_state)

    # 5. 拆分特征和标签
    X_balanced = df_final.drop('label', axis=1).values
    y_balanced = df_final['label'].values

    return X_balanced, y_balanced

# 使用示例
X_balanced, y_balanced = hybrid_resample(X_fs, y, upsample_targets=[5, 6], total_per_class=9689)

In [12]:
pd.Series(y).value_counts()

Attack_type
7     1363998
4      121567
2       67939
11      50826
3       50062
13      50026
8       49933
1       48544
12      36807
0       24026
9       19977
14      15066
10       9689
5         853
6         358
Name: count, dtype: int64

In [13]:
pd.Series(y_balanced).value_counts()

7     1363998
4      121567
2       67939
11      50826
3       50062
13      50026
8       49933
1       48544
12      36807
0       24026
9       19977
14      15066
6        9689
10       9689
5        9689
Name: count, dtype: int64

In [14]:
def stratified_iid_partition_with_test(X, y, num_clients=10, val_ratio=0.1, test_ratio=0.1, random_state=42):
    """
    1. 按 stratified 将原始数据划分为 train / val / test
    2. 再把 train 数据 stratified-IID 地均分到 num_clients 个客户端
    返回:
        client_data: {client_id: (X_client, y_client)}
        test_data:   (X_test, y_test)
        val_data:    (X_val, y_val)
    """
    # ----- 1) Train / Val / Test -----
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=val_ratio + test_ratio, stratify=y, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=test_ratio / (val_ratio + test_ratio),
        stratify=y_temp,
        random_state=random_state
    )

    # ----- 2) 再把 train 分给客户端 -----
    skf = StratifiedKFold(n_splits=num_clients, shuffle=True, random_state=random_state)
    client_data = {}

    for client_id, (_, idx) in enumerate(skf.split(X_train, y_train)):
        client_data[client_id] = (
            X_train[idx],      # features
            y_train[idx],      # labels
        )

    return client_data, (X_test, y_test), (X_val, y_val)


In [15]:
NUM_CLIENTS = 5
client_data, test_data, vali_data = stratified_iid_partition_with_test(X_balanced, y_balanced, num_clients=NUM_CLIENTS)

# 获取测试集
X_test, y_test = test_data

# 获取验证集
X_val, y_val = vali_data

# 查看每个客户端样本数量
for i in range(NUM_CLIENTS):
    print(f"Client {i} size: {len(client_data[i])}")
print(f"Test set size: {len(X_test)}")


Client 0 size: 2
Client 1 size: 2
Client 2 size: 2
Client 3 size: 2
Client 4 size: 2
Test set size: 192784


In [16]:
from collections import Counter

for i in range(len(client_data)):
    _, labels = client_data[i]  # 直接取出标签数组
    label_count = Counter(labels.tolist())  # 转为 list 后计数
    print(f"Client {i} label distribution:")
    for label, count in sorted(label_count.items()):
        print(f"  Class {label}: {count}")
    print()


Client 0 label distribution:
  Class 0: 3844
  Class 1: 7767
  Class 2: 10870
  Class 3: 8010
  Class 4: 19451
  Class 5: 1550
  Class 6: 1550
  Class 7: 218240
  Class 8: 7989
  Class 9: 3196
  Class 10: 1550
  Class 11: 8133
  Class 12: 5889
  Class 13: 8004
  Class 14: 2411

Client 1 label distribution:
  Class 0: 3844
  Class 1: 7767
  Class 2: 10870
  Class 3: 8010
  Class 4: 19451
  Class 5: 1550
  Class 6: 1550
  Class 7: 218240
  Class 8: 7989
  Class 9: 3196
  Class 10: 1550
  Class 11: 8132
  Class 12: 5889
  Class 13: 8005
  Class 14: 2411

Client 2 label distribution:
  Class 0: 3844
  Class 1: 7767
  Class 2: 10871
  Class 3: 8010
  Class 4: 19450
  Class 5: 1550
  Class 6: 1550
  Class 7: 218240
  Class 8: 7989
  Class 9: 3197
  Class 10: 1551
  Class 11: 8132
  Class 12: 5889
  Class 13: 8004
  Class 14: 2410

Client 3 label distribution:
  Class 0: 3845
  Class 1: 7767
  Class 2: 10870
  Class 3: 8010
  Class 4: 19450
  Class 5: 1550
  Class 6: 1550
  Class 7: 218239
  

In [17]:
pd.Series(y_val).value_counts()

7     136400
4      12157
2       6794
11      5082
3       5006
13      5002
8       4993
1       4855
12      3681
0       2403
9       1998
14      1506
10       969
6        969
5        969
Name: count, dtype: int64

In [18]:
pd.Series(y_test).value_counts()

7     136400
4      12157
2       6794
11      5083
3       5006
13      5003
8       4994
1       4854
12      3680
0       2402
9       1997
14      1507
5        969
6        969
10       969
Name: count, dtype: int64

In [19]:
label_encoder.classes_

array(['Backdoor', 'DDoS_HTTP', 'DDoS_ICMP', 'DDoS_TCP', 'DDoS_UDP',
       'Fingerprinting', 'MITM', 'Normal', 'Password', 'Port_Scanning',
       'Ransomware', 'SQL_injection', 'Uploading',
       'Vulnerability_scanner', 'XSS'], dtype=object)

In [20]:
time_start = time.time()
tsne = TSNE(n_components=2, verbose=0, perplexity=40, n_iter=10)
print('t-SNE: {} seconds'.format(time.time()-time_start))

t-SNE: 0.0 seconds


In [21]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight('balanced',
                                                 classes=np.unique(y_balanced),
                                                 y=y_balanced)

class_weights = {k: v for k,v in enumerate(class_weights)}
class_weights

{0: 5.349310469213907,
 1: 2.6475472423643156,
 2: 1.8917342518043148,
 3: 2.567267255270132,
 4: 1.0572156369190104,
 5: 13.264788247841194,
 6: 13.264788247841194,
 7: 0.09422486934242817,
 8: 2.5738996922542876,
 9: 6.433525220670438,
 10: 13.264788247841194,
 11: 2.528676923884101,
 12: 3.491795944611985,
 13: 2.569114727008622,
 14: 8.530634098853932}

In [22]:
input_shape = X_test.shape[1:]

In [23]:
print(X_test.shape)
print(input_shape)

(192784, 91)
(91,)


In [24]:
num_classes = len(np.unique(y_test))
num_classes

15

# Federated Learning

In [25]:
# 创建模型架构
def create_model(input_dim, num_classes):
    model = Sequential()
    model.add(Dense(90, activation='relu', kernel_regularizer=l2(0.01), input_shape=(input_dim,)))
    model.add(Dense(90, activation='relu', kernel_regularizer=l2(0.01)))
    model.add(Dense(num_classes, activation='softmax'))  # softmax 输出
    
    model.compile(optimizer=Adam(),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [26]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("✅ GPU is available:", gpus)
else:
    print("❌ No GPU found, running on CPU.")

✅ GPU is available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [27]:
# Configuration
MODEL_DIR = "aggregator-mpc-urpc1"
CLIENT_MODEL_DIR = "client_weight_mpc-urpc1"
GLOBAL_MODEL_ROUND_DIR = os.path.join(MODEL_DIR, "global_models_per_round")
os.makedirs(GLOBAL_MODEL_ROUND_DIR, exist_ok=True)
os.makedirs(CLIENT_MODEL_DIR, exist_ok=True)

# Hyperparameters
NUM_ROUNDS = 10
EPOCHS = 5
BATCH_SIZE = 256
NOISE_SCALE = 1e-3  # Noise scale for MPC simulation
k=4 # Number of clients to select per round

In [28]:
# Initialize logging
logging.basicConfig(filename='fl_training.log', level=logging.INFO)
logger = logging.getLogger(__name__)

def federated_training(global_model, client_data, X_val, y_val, X_test, y_test, label_encoder):
    """Main federated training loop with secure aggregation simulation."""
    
    # Training history storage
    global_history = {
        'round': [],
        'loss': [],
        'accuracy': [],
        'client_metrics': []
    }

    for round_num in range(NUM_ROUNDS):
        logger.info(f"\n🔁 Federated Round {round_num + 1}/{NUM_ROUNDS}")
        print(f"\n🔁 Federated Round {round_num + 1}/{NUM_ROUNDS}")

        # Select clients for the current round
        selected_indices = np.random.choice(NUM_CLIENTS, k, replace=False)
        total_samples = sum(len(client_data[i][1]) for i in selected_indices)

        # Generate pairwise noise masks for secure aggregation
        noise_masks = generate_noise_masks(global_model, selected_indices)

        # Client training phase
        masked_weights_sum = train_clients(
            global_model, 
            client_data, 
            selected_indices, 
            noise_masks, 
            total_samples,
            X_val, y_val,
            round_num
        )

        # Secure aggregation phase
        aggregated_weights = secure_aggregation(masked_weights_sum, noise_masks)

        # Update global model
        global_model.set_weights(aggregated_weights)
        logger.info("✅ Global model updated.")

        # Save and evaluate
        save_and_evaluate(
            global_model,
            global_history,
            round_num,
            X_test,
            y_test,
            label_encoder
        )

    return global_model, global_history

def generate_noise_masks(global_model, selected_indices) -> Dict[int, List[np.ndarray]]:
    """Generate symmetric noise masks for secure aggregation simulation."""
    noise_masks = {}
    model_weights = global_model.get_weights()
    
    for idx in selected_indices:
        noise_masks[idx] = []
        for layer in model_weights:
            mask = np.random.normal(loc=0.0, scale=NOISE_SCALE, size=layer.shape)
            noise_masks[idx].append(mask)
    
    return noise_masks

def train_clients(global_model, client_data, selected_indices, noise_masks, total_samples, X_val, y_val, round_num):
    """Train clients and collect masked updates."""
    masked_weights_sum = None
    
    for idx in selected_indices:
        logger.info(f"\n📡 Training client {idx + 1}")
        print(f"\n📡 Training client {idx + 1}")

        # Clone global model
        local_model = tf.keras.models.clone_model(global_model)
        local_model.set_weights(global_model.get_weights())
        local_model.compile(
            optimizer=Adam(clipnorm=1.0), 
            loss='sparse_categorical_crossentropy', 
            metrics=['accuracy']
        )

        # Get client data
        X_client, y_client = client_data[idx]

        # Callbacks
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
            ModelCheckpoint(
                os.path.join(CLIENT_MODEL_DIR, f"client_model_round_{round_num+1}_client_{idx}.h5"),
                monitor='val_loss', 
                save_best_only=True, 
                verbose=1
            )
        ]

        # Train
        history = local_model.fit(
            X_client, y_client,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=1,
            callbacks=callbacks
        )

        # Get and mask weights
        local_weights = local_model.get_weights()
        weight_factor = len(y_client) / total_samples
        masked_weights = [
            w * weight_factor + noise_masks[idx][i] 
            for i, w in enumerate(local_weights)
        ]

        # Aggregate masked weights
        if masked_weights_sum is None:
            masked_weights_sum = masked_weights
        else:
            masked_weights_sum = [
                sum_w + new_w 
                for sum_w, new_w in zip(masked_weights_sum, masked_weights)
            ]

        # Clean up
        del local_model
        tf.keras.backend.clear_session()
    
    return masked_weights_sum

def secure_aggregation(masked_weights_sum, noise_masks):
    """Simulate secure aggregation by removing combined noise."""
    # Calculate total noise
    total_noise = [np.zeros_like(w) for w in masked_weights_sum]
    for mask_list in noise_masks.values():
        for i in range(len(total_noise)):
            total_noise[i] += mask_list[i]
    
    # Verify noise cancellation (should sum to zero)
    for i, noise in enumerate(total_noise):
        if not np.allclose(noise, 0, atol=1e-6):
            logger.warning(f"Noise cancellation imperfect in layer {i}")
    
    # Remove noise
    return [
        masked_weights_sum[i] - total_noise[i] 
        for i in range(len(masked_weights_sum))
    ]

def save_and_evaluate(global_model, global_history, round_num, X_test, y_test, label_encoder):
    """Save model and evaluate performance."""
    # Save model
    round_model_path = os.path.join(GLOBAL_MODEL_ROUND_DIR, f"global_model_round_{round_num + 1}.h5")
    global_model.save(round_model_path)
    logger.info(f"💾 Saved global model for round {round_num + 1}")
    
    # Evaluate
    round_loss, round_acc = global_model.evaluate(X_test, y_test, verbose=0)
    global_history['round'].append(round_num + 1)
    global_history['loss'].append(round_loss)
    global_history['accuracy'].append(round_acc)
    
    logger.info(f"🌍 Global Model Evaluation: Loss = {round_loss:.4f}, Accuracy = {round_acc:.4f}")
    print(f"🌍 Global Model Evaluation: Loss = {round_loss:.4f}, Accuracy = {round_acc:.4f}")

    # Generate reports every few rounds
    if (round_num + 1) % 2 == 0:
        generate_reports(global_model, X_test, y_test, label_encoder, round_num)

def generate_reports(model, X_test, y_test, label_encoder, round_num):
    """Generate classification reports and confusion matrix."""
    predictions = model.predict(X_test)
    predicted_classes = np.argmax(predictions, axis=1)
    
    # Classification report
    print("\n📋 Classification Report:")
    print(classification_report(y_test, predicted_classes, 
                              target_names=label_encoder.classes_, 
                              digits=4))
    
    # Confusion matrix
    plt.figure(figsize=(12, 10))
    cm = confusion_matrix(y_test, predicted_classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_, 
                yticklabels=label_encoder.classes_)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"confusion_matrix_round_{round_num + 1}.png")
    plt.close()

# Main execution
if __name__ == "__main__":
    # Initialize your global model
    global_model = create_model(X_test.shape[1], num_classes)
    
    # Run federated training
    trained_model, history = federated_training(
        global_model=global_model,
        client_data=client_data,
        X_val=X_val,
        y_val=y_val,
        X_test=X_test,
        y_test=y_test,
        label_encoder=label_encoder
    )
    
    # Save final model
    final_model_path = os.path.join(MODEL_DIR, "final_global_model.h5")
    trained_model.save(final_model_path)
    logger.info(f"\n💾 Final global model saved to {final_model_path}")


🔁 Federated Round 1/10

📡 Training client 3
Epoch 1/5
1202/1205 [============================>.] - ETA: 0s - loss: 0.5056 - accuracy: 0.9196
Epoch 1: val_loss improved from inf to 0.24123, saving model to client_weight_mpc-urpc1\client_model_round_1_client_2.h5
1205/1205 [==============================] - 4s 3ms/step - loss: 0.5050 - accuracy: 0.9196 - val_loss: 0.2412 - val_accuracy: 0.9275
Epoch 2/5
1186/1205 [============================>.] - ETA: 0s - loss: 0.2253 - accuracy: 0.9283
Epoch 2: val_loss improved from 0.24123 to 0.21473, saving model to client_weight_mpc-urpc1\client_model_round_1_client_2.h5
1205/1205 [==============================] - 3s 3ms/step - loss: 0.2252 - accuracy: 0.9283 - val_loss: 0.2147 - val_accuracy: 0.9275
Epoch 3/5
1198/1205 [============================>.] - ETA: 0s - loss: 0.2084 - accuracy: 0.9280
Epoch 3: val_loss improved from 0.21473 to 0.20168, saving model to client_weight_mpc-urpc1\client_model_round_1_client_2.h5
1205/1205 [================

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



🔁 Federated Round 3/10

📡 Training client 4
Epoch 1/5
1203/1205 [============================>.] - ETA: 0s - loss: 0.1771 - accuracy: 0.9308
Epoch 1: val_loss improved from inf to 0.17592, saving model to client_weight_mpc-urpc1\client_model_round_3_client_3.h5
1205/1205 [==============================] - 5s 4ms/step - loss: 0.1771 - accuracy: 0.9307 - val_loss: 0.1759 - val_accuracy: 0.9319
Epoch 2/5
1205/1205 [==============================] - ETA: 0s - loss: 0.1759 - accuracy: 0.9305
Epoch 2: val_loss did not improve from 0.17592
1205/1205 [==============================] - 4s 4ms/step - loss: 0.1759 - accuracy: 0.9305 - val_loss: 0.1770 - val_accuracy: 0.9313
Epoch 3/5
1196/1205 [============================>.] - ETA: 0s - loss: 0.1748 - accuracy: 0.9309
Epoch 3: val_loss improved from 0.17592 to 0.17451, saving model to client_weight_mpc-urpc1\client_model_round_3_client_3.h5
1205/1205 [==============================] - 4s 4ms/step - loss: 0.1748 - accuracy: 0.9309 - val_loss: 0.

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



🔁 Federated Round 5/10

📡 Training client 2
Epoch 1/5
1202/1205 [============================>.] - ETA: 0s - loss: 0.1702 - accuracy: 0.9310
Epoch 1: val_loss improved from inf to 0.17181, saving model to client_weight_mpc-urpc1\client_model_round_5_client_1.h5
1205/1205 [==============================] - 5s 4ms/step - loss: 0.1702 - accuracy: 0.9311 - val_loss: 0.1718 - val_accuracy: 0.9312
Epoch 2/5
1204/1205 [============================>.] - ETA: 0s - loss: 0.1701 - accuracy: 0.9313
Epoch 2: val_loss improved from 0.17181 to 0.17079, saving model to client_weight_mpc-urpc1\client_model_round_5_client_1.h5
1205/1205 [==============================] - 4s 4ms/step - loss: 0.1701 - accuracy: 0.9313 - val_loss: 0.1708 - val_accuracy: 0.9313
Epoch 3/5
1190/1205 [============================>.] - ETA: 0s - loss: 0.1692 - accuracy: 0.9315
Epoch 3: val_loss improved from 0.17079 to 0.16775, saving model to client_weight_mpc-urpc1\client_model_round_5_client_1.h5
1205/1205 [================

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



🔁 Federated Round 7/10

📡 Training client 5
Epoch 1/5
1196/1205 [============================>.] - ETA: 0s - loss: 0.1671 - accuracy: 0.9314
Epoch 1: val_loss improved from inf to 0.16559, saving model to client_weight_mpc-urpc1\client_model_round_7_client_4.h5
1205/1205 [==============================] - 5s 4ms/step - loss: 0.1672 - accuracy: 0.9314 - val_loss: 0.1656 - val_accuracy: 0.9314
Epoch 2/5
1193/1205 [============================>.] - ETA: 0s - loss: 0.1668 - accuracy: 0.9312
Epoch 2: val_loss did not improve from 0.16559
1205/1205 [==============================] - 4s 3ms/step - loss: 0.1669 - accuracy: 0.9312 - val_loss: 0.1683 - val_accuracy: 0.9309
Epoch 3/5
1187/1205 [============================>.] - ETA: 0s - loss: 0.1664 - accuracy: 0.9314
Epoch 3: val_loss did not improve from 0.16559
1205/1205 [==============================] - 4s 3ms/step - loss: 0.1664 - accuracy: 0.9314 - val_loss: 0.1663 - val_accuracy: 0.9301

📡 Training client 4
Epoch 1/5
1197/1205 [========

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



🔁 Federated Round 9/10

📡 Training client 3
Epoch 1/5
1204/1205 [============================>.] - ETA: 0s - loss: 0.1656 - accuracy: 0.9318
Epoch 1: val_loss improved from inf to 0.16671, saving model to client_weight_mpc-urpc1\client_model_round_9_client_2.h5
1205/1205 [==============================] - 5s 4ms/step - loss: 0.1656 - accuracy: 0.9318 - val_loss: 0.1667 - val_accuracy: 0.9313
Epoch 2/5
1188/1205 [============================>.] - ETA: 0s - loss: 0.1655 - accuracy: 0.9318
Epoch 2: val_loss improved from 0.16671 to 0.16449, saving model to client_weight_mpc-urpc1\client_model_round_9_client_2.h5
1205/1205 [==============================] - 4s 4ms/step - loss: 0.1653 - accuracy: 0.9319 - val_loss: 0.1645 - val_accuracy: 0.9313
Epoch 3/5
1196/1205 [============================>.] - ETA: 0s - loss: 0.1650 - accuracy: 0.9316
Epoch 3: val_loss did not improve from 0.16449
1205/1205 [==============================] - 4s 4ms/step - loss: 0.1651 - accuracy: 0.9316 - val_loss: 0.

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [29]:
# 确保测试集格式正确（true_classes 应为整数标签）
true_classes = y_test  # 如果是 one-hot，则使用 np.argmax(y_test, axis=1)

# 所有标签编号和名称
all_labels = list(range(len(label_encoder.classes_)))
target_names = label_encoder.classes_

# 测试所有 global model
print("\n🌍 Testing Global Models:")
for fname in sorted(os.listdir(GLOBAL_MODEL_ROUND_DIR)):
    if fname.endswith(".h5"):
        model_path = os.path.join(GLOBAL_MODEL_ROUND_DIR, fname)
        model = load_model(model_path)
        pred_probs = model.predict(X_test)
        predicted_classes = np.argmax(pred_probs, axis=1)

        print(f"\n📋 Classification Report for Global Model {fname}:")
        print(classification_report(true_classes, predicted_classes,
                                    labels=all_labels,
                                    target_names=target_names,
                                    digits=4))

# 测试所有 client model
print("\n📡 Testing Client Models:")
for fname in sorted(os.listdir(CLIENT_MODEL_DIR)):
    if fname.endswith(".h5"):
        model_path = os.path.join(CLIENT_MODEL_DIR, fname)
        model = load_model(model_path)
        pred_probs = model.predict(X_test)
        predicted_classes = np.argmax(pred_probs, axis=1)

        print(f"\n📋 Classification Report for Client Model {fname}:")
        print(classification_report(true_classes, predicted_classes,
                                    labels=all_labels,
                                    target_names=target_names,
                                    digits=4))



🌍 Testing Global Models:
6025/6025 [==============================] - 6s 1ms/step

📋 Classification Report for Global Model global_model_round_1.h5:
                       precision    recall  f1-score   support

             Backdoor     0.7208    0.9234    0.8096      2402
            DDoS_HTTP     0.7018    0.9864    0.8201      4854
            DDoS_ICMP     0.9984    0.9480    0.9726      6794
             DDoS_TCP     0.6574    1.0000    0.7933      5006
             DDoS_UDP     0.9802    0.9992    0.9896     12157
       Fingerprinting     0.7075    0.5542    0.6215       969
                 MITM     1.0000    1.0000    1.0000       969
               Normal     1.0000    1.0000    1.0000    136400
             Password     0.4369    0.8833    0.5846      4994
        Port_Scanning     0.0000    0.0000    0.0000      1997
           Ransomware     0.0000    0.0000    0.0000       969
        SQL_injection     0.6362    0.1668    0.2643      5083
            Uploading     0.57

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


6025/6025 [==============================] - 6s 1ms/step

📋 Classification Report for Global Model global_model_round_10.h5:
                       precision    recall  f1-score   support

             Backdoor     0.7208    0.9234    0.8096      2402
            DDoS_HTTP     0.7019    0.9864    0.8202      4854
            DDoS_ICMP     0.9994    0.9769    0.9880      6794
             DDoS_TCP     0.6573    1.0000    0.7932      5006
             DDoS_UDP     0.9887    0.9997    0.9942     12157
       Fingerprinting     0.8333    0.5728    0.6789       969
                 MITM     1.0000    1.0000    1.0000       969
               Normal     1.0000    1.0000    1.0000    136400
             Password     0.4584    0.5481    0.4992      4994
        Port_Scanning     0.0000    0.0000    0.0000      1997
           Ransomware     0.0000    0.0000    0.0000       969
        SQL_injection     0.4581    0.4861    0.4717      5083
            Uploading     0.5870    0.3815    0.4625   

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


6025/6025 [==============================] - 6s 1ms/step

📋 Classification Report for Global Model global_model_round_2.h5:
                       precision    recall  f1-score   support

             Backdoor     0.7208    0.9234    0.8096      2402
            DDoS_HTTP     0.7018    0.9864    0.8201      4854
            DDoS_ICMP     0.9227    0.9906    0.9554      6794
             DDoS_TCP     0.6573    1.0000    0.7932      5006
             DDoS_UDP     0.9861    0.9984    0.9922     12157
       Fingerprinting     0.0000    0.0000    0.0000       969
                 MITM     1.0000    1.0000    1.0000       969
               Normal     1.0000    1.0000    1.0000    136400
             Password     0.4533    0.5102    0.4801      4994
        Port_Scanning     0.0000    0.0000    0.0000      1997
           Ransomware     0.0000    0.0000    0.0000       969
        SQL_injection     0.4525    0.5168    0.4825      5083
            Uploading     0.5761    0.3649    0.4468    

...

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


6025/6025 [==============================] - 6s 1ms/step

📋 Classification Report for Client Model client_model_round_9_client_3.h5:
                       precision    recall  f1-score   support

             Backdoor     0.7208    0.9338    0.8136      2402
            DDoS_HTTP     0.7136    0.9662    0.8209      4854
            DDoS_ICMP     0.9992    0.9706    0.9847      6794
             DDoS_TCP     0.6603    1.0000    0.7954      5006
             DDoS_UDP     0.9905    0.9996    0.9950     12157
       Fingerprinting     0.7741    0.5872    0.6678       969
                 MITM     1.0000    1.0000    1.0000       969
               Normal     1.0000    1.0000    1.0000    136400
             Password     0.4405    0.7505    0.5552      4994
        Port_Scanning     0.0000    0.0000    0.0000      1997
           Ransomware     0.0000    0.0000    0.0000       969
        SQL_injection     0.5054    0.2866    0.3658      5083
            Uploading     0.5824    0.3745    0

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


6025/6025 [==============================] - 7s 1ms/step

📋 Classification Report for Client Model client_model_round_9_client_4.h5:
                       precision    recall  f1-score   support

             Backdoor     0.7208    0.9234    0.8096      2402
            DDoS_HTTP     0.7019    0.9864    0.8202      4854
            DDoS_ICMP     0.9968    0.9762    0.9864      6794
             DDoS_TCP     0.6573    1.0000    0.7932      5006
             DDoS_UDP     0.9884    0.9983    0.9933     12157
       Fingerprinting     0.8246    0.5676    0.6724       969
                 MITM     1.0000    1.0000    1.0000       969
               Normal     1.0000    1.0000    1.0000    136400
             Password     0.8905    0.1482    0.2541      4994
        Port_Scanning     0.0000    0.0000    0.0000      1997
           Ransomware     0.0000    0.0000    0.0000       969
        SQL_injection     0.4339    0.8991    0.5853      5083
            Uploading     0.5873    0.3821    0

c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\anaconda\envs\iiot\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
